In [1]:

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision
from torch.utils.data import DataLoader
import numpy as np
import torch.nn.functional as F

In [ ]:

device=('cuda' if torch.cuda.is_available() else 'cpu')

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])


train_data=torchvision.datasets.CIFAR10(root='./data',train=True,transform=transform_train,download=True)
test_data=torchvision.datasets.CIFAR10(root='./data',train=False,transform=transform_test,download=True)


train_loader=DataLoader(train_data,batch_size=64,shuffle=True)
test_loader=DataLoader(test_data,batch_size=64,shuffle=False)

classess=('plain','card','bird','cat','deer','dog','frog','horse','ship','truck')

class MyConv(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2,2)
        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(64*8*8, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x


model=MyConv().to(device)

loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

model.train()
for i in range(30):
    for images,labels in train_loader:
        images=images.to(device)
        labels=labels.to(device)

        output=model(images)

        loss=loss_fn(output,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f' epoch : {i+1} , loss : {loss.item():.4f}')



0.1%


KeyboardInterrupt: 

In [ ]:
model.eval()
with torch.no_grad():
    total=0
    correct=0
    for images,labels in test_loader:
        images=images.to(device)
        labels=labels.to(device)
        output=model(images)

        _, predicted = torch.max(output, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

acc = 100 * correct / total
print('Accuracy : ' , round(acc,2),'%')


Accuracy :  75.47 %


**`After Improvement`**

In [ ]:

device=('cuda' if torch.cuda.is_available() else 'cpu')

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])


train_data=torchvision.datasets.CIFAR10(root='./data',train=True,transform=transform_train,download=True)
test_data=torchvision.datasets.CIFAR10(root='./data',train=False,transform=transform_test,download=True)


train_loader=DataLoader(train_data,batch_size=64,shuffle=True)
test_loader=DataLoader(test_data,batch_size=64,shuffle=False)

classess=('plain','card','bird','cat','deer','dog','frog','horse','ship','truck')

class MyConv(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2,2)
        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(64*8*8, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x


model=MyConv().to(device)

loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

model.train()
for i in range(30):
    for images,labels in train_loader:
        images=images.to(device)
        labels=labels.to(device)
        
        output=model(images)

        loss=loss_fn(output,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f' epoch : {i+1} , loss : {loss.item():.4f}')


model.eval()
with torch.no_grad():
    total=0
    correct=0
    for images,labels in test_loader:
        images=images.to(device)
        labels=labels.to(device)
        output=model(images)

        _, predicted = torch.max(output, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

acc = 100 * correct / total
print('Accuracy : ' , round(acc,2),'%')

**`Using Resnet18`**

In [ ]:

from torchvision.models import resnet18

# ----------------------------
# Device
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ----------------------------
# Transforms (Data Augmentation)
# ----------------------------
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# ----------------------------
# Dataset
# ----------------------------
train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_train
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_test
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

classes = (
    "plane", "car", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)

# ----------------------------
# Model (ResNet18)
# ----------------------------
model = resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

# ----------------------------
# Loss & Optimizer
# ----------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=50
)

# ----------------------------
# Training
# ----------------------------
epochs = 50
best_acc = 0.0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {running_loss/len(train_loader):.4f}")

    # ----------------------------
    # Evaluation
    # ----------------------------
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"Test Accuracy: {acc:.2f}%")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "best_model.pth")
        print("✅ Model saved")

print(f"\n🔥 Best Accuracy Achieved: {best_acc:.2f}%")


Using device: cuda
Epoch [1/50] Loss: 1.3504
Test Accuracy: 73.81%
✅ Model saved
Epoch [2/50] Loss: 1.1175
Test Accuracy: 73.97%
✅ Model saved
Epoch [3/50] Loss: 1.0472
Test Accuracy: 78.28%
✅ Model saved
Epoch [4/50] Loss: 0.9927
Test Accuracy: 79.16%
✅ Model saved
Epoch [5/50] Loss: 0.9633
Test Accuracy: 80.53%
✅ Model saved
Epoch [6/50] Loss: 0.9559
Test Accuracy: 81.69%
✅ Model saved
Epoch [7/50] Loss: 0.9226
Test Accuracy: 81.14%
Epoch [8/50] Loss: 0.8993
Test Accuracy: 81.02%
Epoch [9/50] Loss: 0.8828
Test Accuracy: 82.74%
✅ Model saved
Epoch [10/50] Loss: 0.9382
Test Accuracy: 81.81%
Epoch [11/50] Loss: 0.8642
Test Accuracy: 83.12%
✅ Model saved
Epoch [12/50] Loss: 0.8484
Test Accuracy: 82.91%
Epoch [13/50] Loss: 0.8341
Test Accuracy: 84.22%
✅ Model saved
Epoch [14/50] Loss: 0.8214
Test Accuracy: 83.34%
Epoch [15/50] Loss: 0.8117
Test Accuracy: 83.96%
Epoch [16/50] Loss: 0.8157
Test Accuracy: 84.59%
✅ Model saved
Epoch [17/50] Loss: 0.7902
Test Accuracy: 85.13%
✅ Model saved
Epo

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import resnet18
from PIL import Image



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

classes = (
    "plane", "car", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)


model = resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 10)
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model = model.to(device)
model.eval()


transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])


image_path = "c1.jpg"   
image = Image.open(image_path).convert("RGB")
image = transform(image).unsqueeze(0).to(device)


with torch.no_grad():
    output = model(image)
    probabilities = torch.softmax(output, dim=1)
    confidence, predicted = torch.max(probabilities, 1)

predicted_class = classes[predicted.item()]
confidence = round(confidence.item() * 100, 2)

print("Predicted Class:", predicted_class)
print("Confidence:", round(confidence.item() * 100, 2), "%")
    